<a href="https://colab.research.google.com/github/seif-gaaloul123/smart-light-controller/blob/main/gg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers --upgrade -q
!pip install scikit-learn xgboost shap imbalanced-learn --upgrade -q

In [ ]:
import kagglehub
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# Direct path to the EXACT file
file_path = os.path.join(path, 'accepted_2007_to_2018Q4.csv.gz')

# Load it (it's gzipped so specify compression)
df = pd.read_csv(file_path,
                 compression='gzip',
                 nrows=100000,
                 low_memory=False)

print(f" Loaded: {df.shape}")

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
th = 0.8
missing_values = df.isna().sum()
cols = missing_values[missing_values > th].index
df_clean = df.drop(columns = cols)
df_clean

In [ ]:
df_clean['loan_status'].unique()

In [ ]:
df_clean = df_clean[df_clean['loan_status'].isin(['Fully Paid','Charged Off'])]
df_clean['target']=(df_clean['loan_status']== 'Charged Off').astype(int)
print((df_clean['target'] ==1).sum())
print(df_clean['target'].mean()*100)
print(df_clean['target'].head(50))
df_clean['target']

In [ ]:
import matplotlib.pyplot as plt

target_counts = df_clean['target'].value_counts()
fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = ['tab:red', 'tab:blue']
ax.bar(['Fully Paid', 'Charged Off'], target_counts, color=bar_colors)
plt.xlabel('Loan Status')
plt.ylabel('Count')
plt.title('Distribution of Loan Status')
plt.show()

In [ ]:
import seaborn as sns
cols = ['loan_amnt', 'int_rate', 'annual_inc', 'target']
corr_matrix = df_clean[cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True,vmin=-1, vmax=1)
ax.set_title('Feature Correlation with Target (Default)', fontsize=14, fontweight='bold', pad=20)
plt.show()

In [ ]:
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

text_columns = ['emp_title', 'title', 'desc', 'purpose']
existing_text_cols = [col for col in text_columns if col in df_clean.columns]
print(f"Available text columns:")
for col in existing_text_cols:
    non_null = df_clean[col].notna().sum()
    pct = non_null / len(df_clean) * 100
    print(f"   ? {col}: {non_null:,} non-null ({pct:.1f}%)")

# Text cleaning function
def clean_text(text):
  text = str(text).lower()
  text = re.sub(r'http\S+', '', text)
  text = re.sub(r'[^a-zA-Z\s]', ' ', text)
  text = re.sub(r'\s+', ' ', text)
  text = text.strip()

  return text

print(" Cleaning and combining text")

df_clean['combined_text'] = ''
for col in existing_text_cols:
    print(f"   Processing {col}...")
    df_clean['combined_text'] += df_clean[col].apply(clean_text) + ' '

df_clean['combined_text'] = df_clean['combined_text'].str.strip()

min_text_length = 10
df_clean = df_clean[df_clean['combined_text'].str.len() >= min_text_length].copy()

print("Text processing complete")
print(f" Rows with sufficient text: {len(df_clean):,}")
print(f" Average text length: {df_clean['combined_text'].str.len().mean():.0f} characters")
print(f" Sample text:")
print(f"'{df_clean['combined_text'].iloc[0][:150]}...'")

# ============================================================================
# STEP 4: NUMERICAL FEATURES
# ============================================================================

print("\n?? STEP 4: Numerical Feature Selection")
print("-"*70)

# Select numerical columns
numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove IDs and target
exclude_cols = ['id', 'member_id', 'target', 'policy_code','recoveries', 'collection_recovery_fee', 'total_rec_late_fee',
    'last_pymnt_amnt', 'last_pymnt_d', 'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int','loan_status','last_fico_range_low','last_fico_range_high']
numerical_cols = [col for col in numerical_cols if col not in exclude_cols]

# Remove columns with too many NaNs
numerical_cols = [col for col in numerical_cols
                  if df_clean[col].notna().sum() / len(df_clean) > 0.5]

print(f" Selected {len(numerical_cols)} numerical features")
print(f"   Examples: {numerical_cols[:10]}")

# Handle missing values in numerical columns (fill with median)
print(f" Handling missing values...")
for col in numerical_cols:
    if df_clean[col].isna().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)

print(f" Missing values filled with median")

# ============================================================================
# STEP 5: PREPARE FEATURES FOR MODELING
# ============================================================================

print("\n  STEP 5: Preparing Final Features")


# Extract features
X_text = df_clean['combined_text'].values
X_numerical = df_clean[numerical_cols].values
y = df_clean['target'].values
df_clean.drop(columns='loan_status', inplace=True)


print(f" Features prepared:")
print(f"   Text features shape: {X_text.shape}")
print(f"   Numerical features shape: {X_numerical.shape}")
print(f"   Target shape: {y.shape}")
print(f"   Default rate: {y.mean()*100:.2f}%")

# ============================================================================
# STEP 6: TRAIN/TEST SPLIT
# ============================================================================

print("\n  STEP 6: Train/Test Split")

X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numerical, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f" Split complete:")
print(f"   Training set: {len(X_text_train):,} samples ({len(X_text_train)/len(X_text)*100:.0f}%)")
print(f"   Test set: {len(X_text_test):,} samples ({len(X_text_test)/len(X_text)*100:.0f}%)")
print(f"   Train default rate: {y_train.mean()*100:.2f}%")
print(f"   Test default rate: {y_test.mean()*100:.2f}%")

scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

In [ ]:
default_texts = df_clean[df_clean['target']==1]['combined_text']
non_default_texts = df_clean[df_clean['target']==0]['combined_text']

print("Sample default text:", default_texts.sample(3, random_state=42).values)
print("Sample non-default text:", non_default_texts.sample(3, random_state=42).values)

In [ ]:
default_counts = default_texts.value_counts().head(10)
non_default_counts = non_default_texts.value_counts().head(10)

print("Top default texts:\n", default_counts)
print("\nTop non-default texts:\n", non_default_counts)


In [ ]:
import numpy as np

for col in df_clean.select_dtypes(include=['int64','float64']):
    corr = np.corrcoef(df_clean[col], df_clean['target'])[0,1]
    if abs(corr) > 0.8:
        print(col, corr)

In [ ]:
leakage_cols = [
    'recoveries', 'collection_recovery_fee', 'total_rec_late_fee',
    'last_pymnt_amnt', 'last_pymnt_d', 'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int','loan_status']
k = []
for i in leakage_cols :
  if i in numerical_cols  :
    k.append(i)
print(k)

In [ ]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel

class BertEmbedding():
  def __init__(self, max_length=128):
    self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {self.device}")
    self.model = BertModel.from_pretrained('bert-base-uncased').to(self.device)
    self.max_length = max_length

  def embeddings(self, text, batch_size=32):
    embedding_tensors = [] # Collect PyTorch tensors
    for i in range(0, len(text), batch_size):
      batch = text[i:i+batch_size].tolist() # Ensure tokenizer receives a list of strings

      input_tokens = self.tokenizer(
          batch,
          return_tensors='pt',
          padding=True,
          truncation=True,
          max_length=self.max_length
      )

      input_tokens = {k: v.to(self.device) for k, v in input_tokens.items()}

      with torch.no_grad():
          output = self.model(**input_tokens).last_hidden_state[:,0,:]
      embedding_tensors.append(output)



    return torch.cat(embedding_tensors, dim=0)

bert = BertEmbedding(max_length = 128)
x_train_bert = bert.embeddings(X_text_train, batch_size=32)
x_test_bert = bert.embeddings(X_text_test, batch_size=32)

# Concatenate numerical and BERT embeddings.
# Move BERT embeddings to CPU and convert to NumPy before concatenating with NumPy arrays.
X_train_bert_combined = np.concatenate([X_num_train_scaled, x_train_bert.cpu().numpy()], axis=1)
X_test_bert_combined = np.concatenate([X_num_test_scaled, x_test_bert.cpu().numpy()], axis=1)
print(f"Shape of combined training data (X_train_bert_combined): {X_train_bert_combined.shape}")
print(f"Shape of combined test data (X_test_bert_combined): {X_test_bert_combined.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=500,  # Top 500 words
    min_df=5,          # Word must appear in at least 5 documents
    max_df=0.8,        # Ignore words in >80% of documents
    ngram_range=(1, 2)) # Unigrams and bigrams
x_train_transform = tfidf.fit_transform(X_text_train)
x_test_transform = tfidf.transform(X_text_test)
x_train_tfidf = np.concatenate([X_num_train_scaled, x_train_transform.toarray()], axis=1)
x_test_tfidf = np.concatenate([X_num_test_scaled, x_test_transform.toarray()], axis=1)

In [ ]:
pip install --upgrade shap


In [ ]:
import shap
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import recall_score, roc_auc_score, confusion_matrix , precision_recall_curve , roc_curve , average_precision_score
import numpy as np
import matplotlib.pyplot as plt

class train_bert_tfidf():
  def __init__(self, x_train, x_text, y_train, y_test):
    self.x_train = x_train
    self.x_text = x_text
    self.y_train = y_train
    self.y_test = y_test
    self.model = None # Initialize model as None

  def tune_train_model(self):
    params = param_grid = {
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}
    negative_samples = (self.y_train == 0).sum()
    positive_samples = (self.y_train == 1).sum()
    scale_pos_weight = negative_samples / positive_samples
    xgboost =xgb.XGBClassifier(
            scale_pos_weight=scale_pos_weight,  # Handle imbalance
            random_state=42,
            eval_metric='auc')
    score = RandomizedSearchCV(xgboost,params,cv=5,scoring='roc_auc')
    score.fit(self.x_train, self.y_train)
    print(score.best_params_)
    self.model = score.best_estimator_ # Assign to self.model


  def predict(self):
    if self.model is None:
      raise ValueError("Model has not been trained. Call tune_train_model first.")
    pred_proba = self.model.predict_proba(self.x_text)[:,1]
    return  pred_proba

  def evaluate(self):
    pred_proba = self.predict()

    precision, recall, thresholds = precision_recall_curve(self.y_test, pred_proba)


    scores = precision[:-1] * recall[:-1]
    threshold = thresholds[np.argmax(scores)]
    pred = (pred_proba >= threshold).astype(int)

    metrics = {
       'threshold': threshold,
       'average_precision_score' : average_precision_score(self.y_test, pred),
       'recall_score': recall_score(self.y_test, pred),
       'roc_auc_score': roc_auc_score(self.y_test, pred_proba)
    }
    return metrics, pred_proba, pred

model_tfidf = train_bert_tfidf(x_train_tfidf, x_test_tfidf, y_train, y_test)
model_tfidf.tune_train_model()
metrics, pred_proba,_ = model_tfidf.evaluate()
print(f"metrics for tfidf model: {metrics}")

model_bert = train_bert_tfidf(X_train_bert_combined, X_test_bert_combined, y_train, y_test)
model_bert.tune_train_model()
metrics, pred_proba,_ = model_bert.evaluate()
print(f"metrics for bert model: {metrics}")

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

class xai():
  def __init__(self, model, x_test, feature_names):
    self.model = model
    self.x_test = x_test
    self.feature_names = feature_names
    self.explainer = shap.TreeExplainer(self.model)
    # Store the full list of shap values for all classes for summary plot
    # self.all_shap_values will be a list of arrays, e.g., [shap_values_class0_all_instances, shap_values_class1_all_instances]
    self.all_shap_values = self.explainer.shap_values(self.x_test)

    # The expected_value for binary classification can be a scalar or a list.
    # We need the base value for the positive class (index 1 if it's a list, or the scalar itself).
    if isinstance(self.explainer.expected_value, list):
        self.expected_value_for_waterfall = self.explainer.expected_value[1]
    else:
        self.expected_value_for_waterfall = self.explainer.expected_value

  def plot_waterfall(self, instance_index=0):
    # Get SHAP values for the specific instance for both classes
    # shap_values_for_single_instance will be a list of arrays, e.g., [shap_values_class0_instance_0, shap_values_class1_instance_0]
    shap_values_for_single_instance = self.explainer.shap_values(self.x_test[instance_index])
    # Select SHAP values for the positive class (class 1) for this instance
    shap_values_class1_for_instance = shap_values_for_single_instance[1]

    # Create a shap.Explanation object for the specific instance and class
    explanation = shap.Explanation(
        values=shap_values_class1_for_instance,
        base_values=self.expected_value_for_waterfall,
        data=self.x_test[instance_index],
        feature_names=self.feature_names
    )
    # Pass the Explanation object to waterfall_plot
    shap.waterfall_plot(explanation, show=False)
    plt.title(f'SHAP Waterfall Plot for Instance {instance_index}')

  def plot_summary(self, ax=None):
    # Pass the full list of SHAP values to summary_plot, and specify class_inds=1 for binary classification
    # self.all_shap_values is already a list of arrays, which summary_plot expects for multi-output
    shap.summary_plot(self.all_shap_values, self.x_test, feature_names=self.feature_names, show=False, class_inds=1)
    plt.title('SHAP Summary Plot (Class 1)')

all_feature_names = numerical_cols + list(tfidf.get_feature_names_out())
plt.figure(figsize=(12, 6))
explainable = xai(model_tfidf.model, x_test_tfidf, all_feature_names)
plt.subplot(1,2,1)
explainable.plot_summary()

plt.subplot(1,2,2)
explainable.plot_waterfall(instance_index=0)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix as sk_confusion_matrix, roc_curve as sk_roc_curve, precision_recall_curve as sk_precision_recall_curve

def plot_confusion_matrix(y_test, pred):
  cm = sk_confusion_matrix(y_test, pred)
  plt.figure()
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
  plt.xlabel('Predicted')
  plt.ylabel('Actual')
  plt.title('Confusion Matrix')
  plt.show()
  return plt.gcf()

def plot_roc_auc_curve(pred_proba, y_test):
  fpr, tpr, thresholds = sk_roc_curve(y_test, pred_proba)
  plt.figure()
  plt.plot(fpr, tpr, label='ROC curve')
  plt.xlabel('False Positive Rate')
  plt.ylabel('True Positive Rate')
  plt.title('ROC Curve')
  plt.show()
  return plt.gcf()

def plot_pr_curve(pred_proba, y_test):
  precision, recall, thresholds = sk_precision_recall_curve(y_test, pred_proba)
  plt.figure()
  plt.plot(recall, precision, label='PR curve')
  plt.xlabel('Recall')
  plt.ylabel('Precision')
  plt.show()
  return plt.gcf()

# Create a new instance of the model trainer
class_calling = train_bert_tfidf(x_train_tfidf, x_test_tfidf, y_train, y_test)

# Initialize and train the model within this new instance
class_calling.tune_train_model()

# Now evaluate the model and get predictions
metrics_new_instance, pred_proba, pred = class_calling.evaluate()
print(metrics_new_instance)

# Plot the curves using the predictions from this instance
cm_plot = plot_confusion_matrix(y_test, pred)
roc_plot = plot_roc_auc_curve(pred_proba, y_test)
pr_plot = plot_pr_curve(pred_proba, y_test)